# ModernFinBERT v2: Benchmark Inference

Evaluate the trained model on held-out benchmarks:
1. Financial PhraseBank (50agree & allagree) — sentence-level
2. FiQA 2018 — sentence-level
3. Twitter Financial News Sentiment — sentence-level
4. FinEntity (SEntFiN) — entity-level sentiment

In [ ]:
!pip install -q transformers datasets peft accelerate scikit-learn 2>/dev/null

In [ ]:
import torch
import numpy as np
import json
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import accuracy_score, f1_score, classification_report

NUM_CLASSES = 3
LABEL_NAMES = ["NEGATIVE", "NEUTRAL", "POSITIVE"]
MAX_LENGTH = 1024
MODEL_NAME = "neoyipeng/ModernFinBERT-v2"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=NUM_CLASSES, attn_implementation="eager",
)
model.to(device)
model.eval()
print(f"Loaded {MODEL_NAME}")

In [ ]:
def run_inference(texts, batch_size=32):
    all_preds = []
    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            batch = texts[i : i + batch_size]
            inputs = tokenizer(
                batch, return_tensors="pt", padding=True,
                truncation=True, max_length=MAX_LENGTH,
            )
            inputs = {k: v.to(device) for k, v in inputs.items()}
            logits = model(**inputs).logits
            preds = torch.argmax(logits, dim=-1).cpu().numpy()
            all_preds.extend(preds)
    return np.array(all_preds)


def evaluate_benchmark(texts, labels, name):
    preds = run_inference(texts)
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average="macro", zero_division=0)
    report = classification_report(
        labels, preds, target_names=LABEL_NAMES,
        output_dict=True, zero_division=0,
    )
    per_class = {c: round(report[c]["f1-score"], 4) for c in LABEL_NAMES}
    print(f"\n{'='*50}")
    print(f"{name} (n={len(texts)})")
    print(f"  Accuracy: {acc:.4f}")
    print(f"  Macro F1: {f1:.4f}")
    print(f"  Per-class F1: {per_class}")
    print(classification_report(labels, preds, target_names=LABEL_NAMES, zero_division=0))
    return {"accuracy": round(acc, 4), "macro_f1": round(f1, 4), "per_class_f1": per_class, "n": len(texts)}

## 1. Financial PhraseBank (sentence-level)

In [ ]:
fpb = load_dataset("financial_phrasebank", "sentences_50agree", trust_remote_code=True)["train"]
results_fpb50 = evaluate_benchmark(fpb["sentence"], fpb["label"], "FPB 50agree")

fpb_all = load_dataset("financial_phrasebank", "sentences_allagree", trust_remote_code=True)["train"]
results_fpb_all = evaluate_benchmark(fpb_all["sentence"], fpb_all["label"], "FPB AllAgree")

## 2. FiQA 2018 (sentence-level, discretized)

In [ ]:
fiqa = load_dataset("pauri32/fiqa-2018", split="train")
fiqa_texts, fiqa_labels = [], []
LO, HI = -0.2, 0.2
for row in fiqa:
    fiqa_texts.append(row["sentence"])
    score = row["sentiment_score"]
    if score < LO:
        fiqa_labels.append(0)
    elif score > HI:
        fiqa_labels.append(2)
    else:
        fiqa_labels.append(1)
results_fiqa = evaluate_benchmark(fiqa_texts, fiqa_labels, "FiQA 2018")

## 3. Twitter Financial News Sentiment (sentence-level)

In [ ]:
tfns = load_dataset("zeroshot/twitter-financial-news-sentiment", split="validation")
tfns_remap = {0: 0, 1: 2, 2: 1}  # Bearish->NEG, Bullish->POS, Neutral->NEU
tfns_labels = [tfns_remap[l] for l in tfns["label"]]
results_tfns = evaluate_benchmark(tfns["text"], tfns_labels, "Twitter Financial News")

## 4. FinEntity / SEntFiN (entity-level sentiment)

Each row has a sentence with multiple entity annotations, each with its own sentiment.
We evaluate per-entity: for each entity span, we feed the full sentence to the model
and compare the model's sentence-level prediction against the entity's gold sentiment.
This tests whether the model captures the dominant entity's sentiment correctly.

In [ ]:
finentity = load_dataset("yixuantt/FinEntity", split="train")
print(f"FinEntity: {len(finentity)} sentences")

ENTITY_LABEL_MAP = {"Positive": 2, "Negative": 0, "Neutral": 1}

# Approach 1: Sentence-level — use majority entity sentiment as gold label
sent_texts, sent_labels = [], []
for row in finentity:
    annotations = row["annotations"]
    if not annotations:
        continue
    label_counts = {}
    for ann in annotations:
        lbl = ENTITY_LABEL_MAP.get(ann["label"], 1)
        label_counts[lbl] = label_counts.get(lbl, 0) + 1
    majority = max(label_counts, key=label_counts.get)
    sent_texts.append(row["content"])
    sent_labels.append(majority)

results_finentity_sent = evaluate_benchmark(
    sent_texts, sent_labels, "FinEntity (sentence-level, majority entity sentiment)"
)

# Approach 2: Per-entity — evaluate each entity annotation independently
# For each entity, construct "[ENTITY]: sentence" as input
entity_texts, entity_labels = [], []
for row in finentity:
    for ann in row["annotations"]:
        entity_name = ann["value"]
        label = ENTITY_LABEL_MAP.get(ann["label"], 1)
        entity_texts.append(f"{entity_name}: {row['content']}")
        entity_labels.append(label)

print(f"\nPer-entity evaluation: {len(entity_texts)} entity-sentence pairs")
print(f"Label distribution: NEG={entity_labels.count(0)}, NEU={entity_labels.count(1)}, POS={entity_labels.count(2)}")

results_finentity_entity = evaluate_benchmark(
    entity_texts, entity_labels, "FinEntity (per-entity, prefixed)"
)

# Approach 3: Multi-entity conflict detection
# Only sentences where entities have DIFFERENT sentiments
conflict_texts, conflict_labels, conflict_entities = [], [], []
for row in finentity:
    annotations = row["annotations"]
    if len(annotations) < 2:
        continue
    labels_in_row = set(ann["label"] for ann in annotations)
    if len(labels_in_row) > 1:
        for ann in annotations:
            conflict_texts.append(f"{ann['value']}: {row['content']}")
            conflict_labels.append(ENTITY_LABEL_MAP.get(ann["label"], 1))
            conflict_entities.append(ann["value"])

print(f"\nConflicting-entity pairs: {len(conflict_texts)}")
if conflict_texts:
    results_finentity_conflict = evaluate_benchmark(
        conflict_texts, conflict_labels, "FinEntity (conflicting entities only)"
    )
else:
    results_finentity_conflict = {"accuracy": 0, "macro_f1": 0, "n": 0}

## Results Summary

In [ ]:
all_results = {
    "fpb_50agree": results_fpb50,
    "fpb_allagree": results_fpb_all,
    "fiqa_2018": results_fiqa,
    "twitter_financial": results_tfns,
    "finentity_sentence": results_finentity_sent,
    "finentity_per_entity": results_finentity_entity,
    "finentity_conflict": results_finentity_conflict,
}

print("\n" + "="*65)
print("RESULTS SUMMARY — ModernFinBERT v2")
print("="*65)
print(f"{'Benchmark':<35} {'Accuracy':>10} {'Macro F1':>10} {'N':>6}")
print("-"*65)
for name, r in all_results.items():
    acc = r.get('accuracy', 0)
    f1 = r.get('macro_f1', 0)
    n = r.get('n', '-')
    print(f"{name:<35} {acc:>10.4f} {f1:>10.4f} {str(n):>6}")

with open("benchmark_results_v2.json", "w") as f:
    json.dump(all_results, f, indent=2)
print("\nSaved to benchmark_results_v2.json")